# EXAONE-3.5-7.8B LoRA 파인튜닝 (RunPod GPU)

`data/final/train.jsonl`(2,808) + `valid.jsonl`(311) 로 EXAONE 플래너를 LoRA 파인튜닝한다.
학습 로직은 `sft_pipeline/train/train_plain.py`(transformers Trainer + peft) 가 담당한다.

> **왜 train_plain 인가**: unsloth 는 EXAONE(`custom_code` 아키텍처)를 지원하지 않아
> `No config file found` 로 죽는다. 표준 transformers 스택으로 직접 학습한다.

**전제**: RunPod *PyTorch* 템플릿(CUDA 12.x, **VRAM 24GB+** = RTX 3090/4090/A5000 이상).
리포는 `/workspace/mongle-ai` 에 clone, `feat/exaone-planner-finetune` 체크아웃.
데이터는 S3(`$SFT_BUCKET/$SFT_PREFIX`)에서 받는다(로컬에서 미리 `aws s3 cp` 로 올려둘 것).

In [ ]:
# 0. GPU 확인 (VRAM 24GB+ 여야 EXAONE-7.8B QLoRA 4bit 가능)
!nvidia-smi

In [ ]:
# 1. 리포 위치로 이동 (RunPod 볼륨 경로에 맞게 수정)
%cd /workspace/mongle-ai
!ls sft_pipeline/data/final/ 2>/dev/null || echo '데이터 아직 없음 — 셀 3 에서 S3 다운로드'

In [ ]:
# 2. 의존성 설치 — unsloth/trl 불필요(train_plain 은 표준 스택).
#    transformers 5.x 고정: EXAONE custom 코드가 modeling_rope_utils.RopeParameters(5.x 전용)를 import.
!pip install -q 'transformers==5.5.0' 'huggingface_hub>=0.30' peft bitsandbytes accelerate datasets awscli

In [ ]:
# 2.5 HuggingFace 토큰 — EXAONE 다운로드용. Read 토큰 발급: huggingface.co/settings/tokens
#     login() 이 토큰을 ~/.cache 에 저장 → 이후 학습 서브프로세스도 인증됨.
import os
from huggingface_hub import login

os.environ['HF_TOKEN'] = 'hf_여기에_실제_토큰'  # <-- 교체
login(token=os.environ['HF_TOKEN'])

In [ ]:
# 3. 데이터를 S3에서 받기 (data/final 이 비어있을 때만 필요)
#    로컬에서 미리 올려둔다:
#      aws s3 cp sft_pipeline/data/final/train.jsonl s3://$BUCKET/$PREFIX/train.jsonl
#      aws s3 cp sft_pipeline/data/final/valid.jsonl s3://$BUCKET/$PREFIX/valid.jsonl
#    AWS 키는 pod env 또는 아래에 직접 설정:
#      os.environ['AWS_ACCESS_KEY_ID'] = '...'; os.environ['AWS_SECRET_ACCESS_KEY'] = '...'
import os

BUCKET = os.environ.get('SFT_BUCKET', 'mongle-village-prod-962214557220-ap-northeast-2-an')
PREFIX = os.environ.get('SFT_PREFIX', 'mongle-village/sft/datasets/planner')
os.environ.setdefault('AWS_DEFAULT_REGION', 'ap-northeast-2')

!mkdir -p sft_pipeline/data/final
!aws s3 cp s3://{BUCKET}/{PREFIX}/train.jsonl sft_pipeline/data/final/train.jsonl
!aws s3 cp s3://{BUCKET}/{PREFIX}/valid.jsonl sft_pipeline/data/final/valid.jsonl
!ls -lh sft_pipeline/data/final/

In [ ]:
# 4. 데이터 정합성 확인 — 줄 수 + JSON 파싱 + messages 키 + kind 분포
#    kind 는 planner_chat 노드에만 있는 필드 → '?' 다수는 정상(splitter/goal_tag 등 다른 노드).
import json
from pathlib import Path

for name in ['train', 'valid']:
    path = Path(f'sft_pipeline/data/final/{name}.jsonl')
    rows = [json.loads(l) for l in path.open() if l.strip()]
    assert all('messages' in r for r in rows), f'{name}: messages 키 누락'
    kinds = {}
    for r in rows:
        try:
            k = json.loads(r['messages'][-1]['content']).get('kind', '?')
        except Exception:
            k = 'non_json'
        kinds[k] = kinds.get(k, 0) + 1
    print(f'{name}: {len(rows)}개 | kind 분포: {kinds}')

In [ ]:
# 5. LoRA 학습 — tmux 세션에 "분리 실행"하여 연결이 끊겨도(맥북 덮어도) 계속 돌게 한다.
#    노트북 !python 직접 실행은 커널/브라우저 끊김 시 학습까지 죽으므로 금지.
#    끝나면 outputs/exaone-planner-lora/ 에 어댑터 + postcheck_report.json 저장.
import subprocess

# tmux 보장 (없으면 설치)
subprocess.run("command -v tmux >/dev/null || (apt-get update && apt-get install -y tmux)", shell=True)

# 학습 명령 — VRAM 24GB 안전값(batch 1·grad-accum 8 = effective 8). epoch 1 권장(1회독에 loss 바닥).
# HF_HUB_OFFLINE 은 모델이 이미 캐시된 "재실행"용. 새 Pod 첫 다운로드 시엔 셀 2.5(토큰) 먼저 돌리고
# 아래 export 줄에서 OFFLINE 두 개를 빼고 실행할 것.
TRAIN_CMD = (
    "cd /workspace/mongle-ai && "
    "export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 && "
    "python -m sft_pipeline.train.train_plain "
    "--train sft_pipeline/data/final/train.jsonl "
    "--valid sft_pipeline/data/final/valid.jsonl "
    "--out outputs/exaone-planner-lora "
    "--model LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct "
    "--epochs 1 --batch 1 --grad-accum 8 "
    "2>&1 | tee /workspace/train.log"
)
subprocess.run("tmux kill-session -t train 2>/dev/null", shell=True)
subprocess.run(["tmux", "new-session", "-d", "-s", "train", TRAIN_CMD])
print("✅ 학습 시작됨 (tmux 세션: train). 맥북 덮어도/연결 끊겨도 계속 돕니다.")
print("   진행 확인: 아래 모니터링 셀(5b)을 아무 때나 반복 실행.")
print("   완료 신호: train.log 마지막에 '[train] LoRA 어댑터 저장' + '점검 리포트'.")


In [ ]:
# 5b. 학습 모니터링 — 아무 때나 반복 실행 가능 (학습엔 영향 없음).
#     '[train] LoRA 어댑터 저장' 이 보이면 완료 → 셀 6·7 로 진행.
#     세션 직접 보기: 터미널에서 `tmux attach -t train` (분리는 Ctrl+B → D).
!tail -30 /workspace/train.log

In [ ]:
# 7. 스모크 테스트 — base + 어댑터로 valid 1건 생성해 JSON 파싱 확인(표준 transformers + peft).
import os
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.pop("HF_TOKEN", None)  # 빈 토큰이 Bearer 헤더를 깨뜨리는 문제 방지

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sft_pipeline.train.train_plain import to_ids  # 토큰화 반환형 정규화

BASE = 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct'
ADAPTER = 'outputs/exaone-planner-lora'

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map={'': 0},
    torch_dtype=torch.bfloat16, trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

sample = json.loads(open('sft_pipeline/data/final/valid.jsonl').readline())
prompt_msgs = [m for m in sample['messages'] if m['role'] != 'assistant']
ids = torch.tensor(
    [to_ids(tok.apply_chat_template(prompt_msgs, tokenize=True, add_generation_prompt=True))],
    device=model.device,
)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=512, do_sample=False)
text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
print(text)
print('\n--- JSON 파싱 ---')
try:
    print('OK' if json.loads(text) else 'FAIL')
except Exception as e:
    print('FAIL:', e)


In [ ]:
# 7. 스모크 테스트 — base + 어댑터로 valid 1건 생성해 JSON 파싱 확인(표준 transformers + peft).
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sft_pipeline.train.train_plain import to_ids  # 토큰화 반환형 정규화

BASE = 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct'
ADAPTER = 'outputs/exaone-planner-lora'

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map={'': 0},
    torch_dtype=torch.bfloat16, trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

sample = json.loads(open('sft_pipeline/data/final/valid.jsonl').readline())
prompt_msgs = [m for m in sample['messages'] if m['role'] != 'assistant']
ids = torch.tensor(
    [to_ids(tok.apply_chat_template(prompt_msgs, tokenize=True, add_generation_prompt=True))],
    device=model.device,
)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=1024, do_sample=False)
text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
print(text)
print('\n--- JSON 파싱 ---')
try:
    print('OK' if json.loads(text) else 'FAIL')
except Exception as e:
    print('FAIL:', e)


## 다음 단계

- `outputs/exaone-planner-lora/postcheck_report.json` 의 `parse_success_rate` 가 핵심 지표
  (eval_loss 는 출력이 고정 JSON 이라 자연히 낮아서 과적합 판정 기준 아님).
- 어댑터를 S3/HF 로 업로드 후 서빙(RunPod LLM 워커)의 LoRA repo 로 배선:
  ```bash
  cd outputs && tar czf adapter.tgz exaone-planner-lora
  aws s3 cp adapter.tgz s3://$BUCKET/mongle-village/sft/adapters/exaone-planner/adapter.tgz
  ```
- **Pod 은 ephemeral** — 어댑터 회수 전 terminate 금지.
- 정량 평가는 `sft_pipeline/eval/planner_loop_eval.ipynb` 참고.